##### 1. Setup Environment & Imports
Cài đặt môi trường và nhập các thư viện cần thiết

In [11]:
# Import thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore')

# Cấu hình Pandas display
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("Thư viện đã được nhập thành công")

Thư viện đã được nhập thành công


##### 2. Tìm kiếm và đọc file CSV

In [12]:
# Lấy đường dẫn thư mục hiện tại
current_dir = Path.cwd()
base_dir = Path(r"d:\My Stuff\KHDL28B\Deep Learning")
work_dir = base_dir / 'data'

print(f"Thư mục dữ liệu: {work_dir}")
print(f"Đang sử dụng file: raw_data.csv\n")

csv_path = work_dir / 'raw_data.csv'

# Đọc file CSV - Bỏ qua metadata
print("Đang đọc file CSV...\n")

try:
    # File này có metadata ở 2 dòng đầu, nên bỏ qua dòng 0-2
    df = pd.read_csv(csv_path, skiprows=3)
    
    print("Đã đọc file CSV thành công!")
    print("\nThông tin cơ bản về dữ liệu:\n")
    print(f"Kích thước: {df.shape[0]} hàng × {df.shape[1]} cột")
    
except Exception as e:
    print(f"Lỗi: {e}")

Thư mục dữ liệu: d:\My Stuff\KHDL28B\Deep Learning\data
Đang sử dụng file: raw_data.csv

Đang đọc file CSV...

Đã đọc file CSV thành công!

Thông tin cơ bản về dữ liệu:

Kích thước: 92640 hàng × 13 cột


##### 3. Xem dữ liệu ban đầu

In [13]:
df

,time,pm10 (μg/m³),pm2_5 (μg/m³),carbon_monoxide (μg/m³),carbon_dioxide (ppm),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³),methane (μg/m³),uv_index_clear_sky (),uv_index (),dust (μg/m³),aerosol_optical_depth ()
0,2016-01-01T00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-01-01T01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-01-01T02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-01-01T03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-01-01T04:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
92635,2026-07-26T19:00,12.9,10.5,291.0,451.0,3.3,1.5,65.0,1417.0,0.0,0.0,4.0,0.21
92636,2026-07-26T20:00,12.7,10.6,320.0,455.0,3.6,1.5,59.0,1433.0,0.0,0.0,4.0,0.20
92637,2026-07-26T21:00,12.4,10.4,343.0,459.0,4.0,1.6,50.0,1448.0,0.0,0.0,3.0,0.18
92638,2026-07-26T22:00,12.1,10.2,345.0,461.0,4.3,1.6,43.0,1454.0,0.0,0.0,3.0,0.17


##### 4. Kiểm tra thông tin chi tiết về cột dữ liệu

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92640 entries, 0 to 92639
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   time                      92640 non-null  object 
 1   pm10 (μg/m³)              34865 non-null  float64
 2   pm2_5 (μg/m³)             34865 non-null  float64
 3   carbon_monoxide (μg/m³)   34865 non-null  float64
 4   carbon_dioxide (ppm)      15329 non-null  float64
 5   nitrogen_dioxide (μg/m³)  34865 non-null  float64
 6   sulphur_dioxide (μg/m³)   34865 non-null  float64
 7   ozone (μg/m³)             34865 non-null  float64
 8   methane (μg/m³)           15329 non-null  float64
 9   uv_index_clear_sky ()     34865 non-null  float64
 10  uv_index ()               34865 non-null  float64
 11  dust (μg/m³)              34865 non-null  float64
 12  aerosol_optical_depth ()  34865 non-null  float64
dtypes: float64(12), object(1)
memory usage: 9.2+ MB


##### 5. Loại bỏ các hàng có toàn bộ giá trị NaN (trừ cột time)

In [15]:
cols_to_check = [col for col in df.columns if col != 'time']
df_copy = df.dropna(subset=cols_to_check, how='all')

##### 6. Loại bỏ các cột có chứa giá trị NaN

In [16]:
df_copy = df_copy.dropna(axis=1)

##### 7. Chuyển đổi cột time thành datetime và tách thành các cột ngày, tháng, năm, giờ

In [17]:
df_copy['time'] = pd.to_datetime(df_copy['time'])
df_copy['day'] = df_copy['time'].dt.day
df_copy['month'] = df_copy['time'].dt.month
df_copy['year'] = df_copy['time'].dt.year
df_copy['hour'] = df_copy['time'].dt.hour

cols = df_copy.columns.tolist()
time_idx = cols.index('time')
cols = cols[:time_idx+1] + ['day', 'month', 'year', 'hour'] + [c for c in cols[time_idx+1:] if c not in ['day', 'month', 'year', 'hour']]
df_copy = df_copy[cols]
df_copy.reset_index(drop=True, inplace=True)

##### 8. Xem dữ liệu sau khi làm sạch và chuyển đổi

In [18]:
df_copy

,time,day,month,year,hour,pm10 (μg/m³),pm2_5 (μg/m³),carbon_monoxide (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³),uv_index_clear_sky (),uv_index (),dust (μg/m³),aerosol_optical_depth ()
0,2022-08-04 07:00:00,4,8,2022,7,14.4,10.1,196.0,2.0,0.6,35.0,1.20,1.05,0.0,0.09
1,2022-08-04 08:00:00,4,8,2022,8,11.1,7.8,186.0,1.6,0.5,38.0,3.55,3.05,0.0,0.09
2,2022-08-04 09:00:00,4,8,2022,9,9.4,6.5,172.0,1.1,0.4,41.0,6.95,5.95,0.0,0.08
3,2022-08-04 10:00:00,4,8,2022,10,10.0,7.0,160.0,0.7,0.4,47.0,10.45,8.50,0.0,0.09
4,2022-08-04 11:00:00,4,8,2022,11,12.1,8.5,161.0,0.7,0.5,51.0,12.75,10.10,0.0,0.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34860,2026-07-26 19:00:00,26,7,2026,19,12.9,10.5,291.0,3.3,1.5,65.0,0.00,0.00,4.0,0.21
34861,2026-07-26 20:00:00,26,7,2026,20,12.7,10.6,320.0,3.6,1.5,59.0,0.00,0.00,4.0,0.20
34862,2026-07-26 21:00:00,26,7,2026,21,12.4,10.4,343.0,4.0,1.6,50.0,0.00,0.00,3.0,0.18
34863,2026-07-26 22:00:00,26,7,2026,22,12.1,10.2,345.0,4.3,1.6,43.0,0.00,0.00,3.0,0.17


##### 9. Kiểm tra xem dữ liệu time có liên tục không (cách 1 giờ)

In [19]:
def check_continuous_time(df):
    if 'time' not in df.columns:
        return False
    
    time_series = pd.to_datetime(df['time'])
    time_diff = time_series.diff()
    expected_diff = pd.Timedelta(hours=1)
    
    is_continuous = (time_diff.iloc[1:] == expected_diff).all()
    return is_continuous

check_continuous_time(df_copy)

True

##### 10. Lưu dữ liệu đã làm sạch thành file CSV mới

In [20]:
output_path = work_dir / 'cleaned_data.csv'
df_copy.to_csv(output_path, index=False, encoding='utf-8')
print(f"Dữ liệu đã được lưu tại: {output_path}")

Dữ liệu đã được lưu tại: d:\My Stuff\KHDL28B\Deep Learning\data\cleaned_data.csv
